In [1]:
import pandas as pd
import numpy as np
import os
import json
from scipy.stats import t

INPUT_PATH = "wikidata_pessoa_homonimo.jsonl"
OUTPUT_BASE = "splits"
NUM_FOLDS = 5

os.makedirs(OUTPUT_BASE, exist_ok=True)

# pega uma linha que contém uma lista de homônimos e a transforma em múltiplas linhas com candidatos
def processar_pares(df_aninhado):
    df_exp = df_aninhado.explode('Lista_homonimos').reset_index(drop=True)
    res = [
        (1 if p1.get('id') == c.get('id') else 0, p1.get('id'), c.get('id'))
        if isinstance(p1, dict) and isinstance(c, dict) else (0, None, None)
        for p1, c in zip(df_exp['Dados_pessoa_1'], df_exp['Lista_homonimos'])
    ]
    df_exp['label'], df_exp['id_pessoa'], df_exp['id_candidato'] = zip(*res)
    
    df_exp = df_exp.rename(columns={
        'id': 'id_query_original', 'fold': 'fold_origem',
        'Dados_pessoa_1': 'pessoa_ground_truth', 'Dados_pessoa_2': 'pessoa_contexto',
        'Lista_homonimos': 'candidato', 'Parentesco': 'parentesco'
    })
    
    cols_order = ['id_query_original', 'fold_origem', 'id_pessoa', 'id_candidato', 
                  'pessoa_ground_truth', 'pessoa_contexto', 'candidato', 'parentesco', 'label']
    return df_exp[cols_order]

print("Bibliotecas importadas.")

Bibliotecas importadas.


In [2]:
df_master = pd.read_json(INPUT_PATH, lines=True)

colunas_limpar = [col for col in ['id', 'fold'] if col in df_master.columns]
if colunas_limpar: df_master = df_master.drop(columns=colunas_limpar)

# mistura os dados e divide o índice do dataframe em 5 partes iguais
df_master = df_master.sample(frac=1, random_state=42).reset_index(drop=True)
df_master['id'] = range(1, len(df_master) + 1)
df_master['fold'] = pd.qcut(df_master.index, q=NUM_FOLDS, labels=range(1, NUM_FOLDS + 1))

df_master.to_json(INPUT_PATH, orient='records', lines=True, force_ascii=False)

estatisticas_gerais = []

print("Dados originais preparados.")

Dados originais preparados.


In [3]:
def calcular_prioridade(row):
    p1 = row['pessoa_ground_truth']
    c = row['candidato']

    if not isinstance(p1, dict) or not isinstance(c, dict):
        return 16

    def has_val(d, field):
        val = d.get(field)
        if isinstance(val, list): 
            return len(val) > 0
        return bool(val)

    has_E = has_val(p1, 'endereço') and has_val(c, 'endereço')
    has_N = has_val(p1, 'data_nascimento') and has_val(c, 'data_nascimento')
    has_Na = has_val(p1, 'nacionalidade') and has_val(c, 'nacionalidade')
    has_O = has_val(p1, 'profissao') and has_val(c, 'profissao')

    if has_E and has_N and has_Na and has_O: return 1
    if has_E and has_N and has_O: return 2
    if has_E and has_Na and has_O: return 3
    if has_E and has_N and has_Na: return 4
    if has_N and has_Na and has_O: return 5
    if has_E and has_O: return 6
    if has_E and has_N: return 7
    if has_E and has_Na: return 8
    if has_N and has_O: return 9
    if has_Na and has_O: return 10
    if has_N and has_Na: return 11
    if has_E: return 12
    if has_O: return 13
    if has_N: return 14
    if has_Na: return 15
    return 16

def aplicar_balanceamento(df_train, num_negativos=10):
    df_pos = df_train[df_train['label'] == 1].copy()
    df_neg = df_train[df_train['label'] == 0].copy()

    df_neg['prioridade'] = df_neg.apply(calcular_prioridade, axis=1)
    df_neg = df_neg.sample(frac=1, random_state=42).sort_values(by=['id_query_original', 'prioridade'])

    df_neg_top = df_neg.groupby('id_query_original').head(num_negativos)
    df_neg_top = df_neg_top.drop(columns=['prioridade'])

    df_balanced = pd.concat([df_pos, df_neg_top])
    df_balanced = df_balanced.sort_values(by=['id_query_original', 'label'], ascending=[True, False]).reset_index(drop=True)
    
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

def gerar_stats_do_treino(fold_num):
    path_treino = f"{OUTPUT_BASE}/wikidata_split{fold_num}_train.jsonl"
    df = pd.read_json(path_treino, lines=True)
    
    df = df[df['label'] == 1].copy()

    def get_year(p):
        try: return int(str(p.get('data_nascimento'))[:4])
        except: return None

    df['ano_alvo'] = df['pessoa_ground_truth'].apply(get_year)
    df['ano_parente'] = df['pessoa_contexto'].apply(get_year)
    df['diff'] = df['ano_alvo'] - df['ano_parente']
    
    # estima os três parâmetros pedidos
    def estimar_t_student(series):
        if len(series) > 1:
            graus_liberdade, localizacao, escala = t.fit(series)
            if escala <= 0: escala = 1.0 
        elif len(series) == 1:
            graus_liberdade, localizacao, escala = 1.0, series.iloc[0], 1.0
        else:
            graus_liberdade, localizacao, escala = 1.0, 0.0, 1.0
            
        return pd.Series({'df': graus_liberdade, 'loc': localizacao, 'scale': escala})

    df['parentesco_lower'] = df['parentesco'].astype(str).str.lower()
    
    stats = df.dropna(subset=['diff']).groupby('parentesco_lower')['diff'].apply(estimar_t_student).unstack().to_dict('index')
    
    with open(f"{OUTPUT_BASE}/stats_split{fold_num}.json", 'w') as f:
        json.dump(stats, f)

print("Funções preparadas.")

Funções preparadas.


In [4]:
# teste fica com linhas do arquivo original e treino/validação pares isolados nas linhas
print("Iniciando processamento para a geração de folds...\n")

for fold_num in range(1, NUM_FOLDS + 1):    
    df_test = df_master[df_master['fold'] == fold_num].copy() # seleciona como teste tudo o que é o fold atual
    df_test.to_json(f"{OUTPUT_BASE}/wikidata_split{fold_num}_test.jsonl", orient='records', lines=True, force_ascii=False)

    df_train_val_source = df_master[df_master['fold'] != fold_num].copy() # tudo o que não é o fold atual
    queries_treino = df_train_val_source.sample(frac=0.9, random_state=42) # 90% treino
    queries_validacao = df_train_val_source.drop(queries_treino.index) # 10% validação

    train_split = processar_pares(queries_treino)
    val_split = processar_pares(queries_validacao)

    train_split = aplicar_balanceamento(train_split, num_negativos=10) # classe de balanceamento
    
    train_split.to_json(f"{OUTPUT_BASE}/wikidata_split{fold_num}_train.jsonl", orient='records', lines=True, force_ascii=False)
    val_split.to_json(f"{OUTPUT_BASE}/wikidata_split{fold_num}_validation.jsonl", orient='records', lines=True, force_ascii=False)

    gerar_stats_do_treino(fold_num)
    
    # treino
    t_qtd_data = train_split['id_query_original'].nunique()
    t_qtd_pairwise = len(train_split)
    t_label_1 = train_split['label'].sum()
    t_label_0 = t_qtd_pairwise - t_label_1
    
    # validação
    v_qtd_data = val_split['id_query_original'].nunique()
    v_tamanhos = val_split.groupby('id_query_original').size()
    v_media = v_tamanhos.mean()
    v_desvio = v_tamanhos.std() if len(v_tamanhos) > 1 else 0.0
    
    # teste
    test_qtd_data = len(df_test)
    test_tamanhos = df_test['Lista_homonimos'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    test_media = test_tamanhos.mean()
    test_desvio = test_tamanhos.std() if len(test_tamanhos) > 1 else 0.0
    
    estatisticas_gerais.append({
        'Fold': fold_num,
        'Treino_Qtd_Data': t_qtd_data,
        'Treino_Pairwise_Total': t_qtd_pairwise,
        'Treino_Label_1': t_label_1,
        'Treino_Label_0': t_label_0,
        'Val_Qtd_Data': v_qtd_data,
        'Val_Media_Homonimos': round(v_media, 2),
        'Val_Desvio_Homonimos': round(v_desvio, 2),
        'Teste_Qtd_Data': test_qtd_data,
        'Teste_Media_Homonimos': round(test_media, 2),
        'Teste_Desvio_Homonimos': round(test_desvio, 2)
    })

    print(f"Splits para o fold {fold_num} gerados.")

print("\nProcesso finalizado.")

Iniciando processamento para a geração de folds...

Splits para o fold 1 gerados.
Splits para o fold 2 gerados.
Splits para o fold 3 gerados.
Splits para o fold 4 gerados.
Splits para o fold 5 gerados.

Processo finalizado.


In [5]:
# exibição do relatório final
df_estatisticas = pd.DataFrame(estatisticas_gerais)
df_estatisticas.set_index('Fold', inplace=True)

print("SPLITS: Resultados Processamento das bases para o artigo SBBD 2026".center(80))
print("\n[TREINO]")
print(df_estatisticas[['Treino_Qtd_Data', 'Treino_Pairwise_Total', 'Treino_Label_1', 'Treino_Label_0']].to_string())

print("\n[VALIDAÇÃO]")
print(df_estatisticas[['Val_Qtd_Data', 'Val_Media_Homonimos', 'Val_Desvio_Homonimos']].to_string())

print("\n[TESTE]")
print(df_estatisticas[['Teste_Qtd_Data', 'Teste_Media_Homonimos', 'Teste_Desvio_Homonimos']].to_string())
print("\nProcesso finalizado.")

       SPLITS: Resultados Processamento das bases para o artigo SBBD 2026       

[TREINO]
      Treino_Qtd_Data  Treino_Pairwise_Total  Treino_Label_1  Treino_Label_0
Fold                                                                        
1              190444                 900515          190444          710071
2              190445                 901167          190445          710722
3              190445                 900648          190445          710203
4              190445                 899570          190445          709125
5              190444                 900189          190444          709745

[VALIDAÇÃO]
      Val_Qtd_Data  Val_Media_Homonimos  Val_Desvio_Homonimos
Fold                                                         
1            21161                11.44                 44.73
2            21161                11.78                 47.57
3            21161                11.51                 45.73
4            21161                11.53        